In [21]:
import random
import uuid
from datetime import datetime, timedelta
import pandas as pd
from faker import Faker
from sqlalchemy import create_engine, inspect
import sqlite3

fake = Faker()

def simulate_data(start_date, end_date, n_users):
    # Convert input dates
    start_date = datetime.strptime(start_date, "%Y-%m-%d")
    end_date = datetime.strptime(end_date, "%Y-%m-%d")

    # Transition map (estado -> [(siguiente_estado, probabilidad)])
    transitions = {
        "sesion_start": [
            ("sesion_start", 0.20),
            ("search_page", 0.80)
        ],
        "search_page": [
            ("search_page", 0.05),
            ("sesion_start", 0.27),
            ("product_page", 0.68),
        ],
        "product_page": [
            ("product_page", 0.15),
            ("car_checkout", 0.58),
            ("sesion_start", 0.27),
        ],
        "car_checkout": [
            ("car_checkout", 0.16),
            ("payment_confirmation", 0.57),
            ("sesion_start", 0.27),
        ],
        "payment_confirmation": [
            ("payment_confirmation", 0.35),
            ("sesion_start", 0.65),
        ]
    }

    def choose_next(state):
        next_states, probs = zip(*transitions[state])
        return random.choices(next_states, probs, k=1)[0]

    users = []
    events = []

    for _ in range(n_users):
        user_id = str(uuid.uuid4())
        first_visit = fake.date_time_between(start_date=start_date, end_date=end_date)

        user = {
            "id": user_id,
            "first_visit": first_visit,
            "first_name": fake.first_name(),
            "last_name": fake.last_name(),
            "sex": random.choice(["F", "M"]),
            "age": random.randint(18, 62),
            "platform": random.choice(["web", "mobile"]),
            "zone": random.choice(["North", "South", "West", "East"])
        }
        users.append(user)

        # Generación de eventos
        session_count = 0
        current_state = "sesion_start"
        current_time = first_visit

        while session_count < 8:
            events.append({
                "id": str(uuid.uuid4()),
                "user_id": user_id,
                "event_date": current_time,
                "event_name": current_state
            })

            next_state = choose_next(current_state)

            # Loop → terminar
            if next_state == current_state:
                break

            # Si el usuario vuelve a sesion_start → sumar sesión y usar retorno largo
            if next_state == "sesion_start":
                session_count += 1
                if session_count >= 10:
                    break
                # Tiempo entre 18 y 200 horas (retorno natural)
                current_time += timedelta(hours=random.uniform(18, 200))
            else:
                # Tiempo entre eventos normales: 18 a 900 segundos
                current_time += timedelta(seconds=random.randint(18, 900))

            current_state = next_state

    users_df = pd.DataFrame(users)
    events_df = pd.DataFrame(events).sort_values(by=["user_id", "event_date"])

    return users_df, events_df



In [34]:
users, events = simulate_data("2025-01-01", "2025-04-30", 2131)


In [35]:
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2131 entries, 0 to 2130
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   id           2131 non-null   object        
 1   first_visit  2131 non-null   datetime64[ns]
 2   first_name   2131 non-null   object        
 3   last_name    2131 non-null   object        
 4   sex          2131 non-null   object        
 5   age          2131 non-null   int64         
 6   platform     2131 non-null   object        
 7   zone         2131 non-null   object        
dtypes: datetime64[ns](1), int64(1), object(6)
memory usage: 133.3+ KB


In [36]:
events.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13634 entries, 4777 to 4166
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   id          13634 non-null  object        
 1   user_id     13634 non-null  object        
 2   event_date  13634 non-null  datetime64[ns]
 3   event_name  13634 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 532.6+ KB


In [38]:
 # ========================
# 2️⃣ Crear conexión SQLite nueva base
# ========================
sqlite_filename = "user_journey.db"
sqlite_conn = sqlite3.connect(sqlite_filename)

In [39]:
for table,df in {'events':events,'users':users}.items():
        df.to_sql(table, sqlite_conn, if_exists='replace', index=False)
        print(f"✅ Tabla '{table}' copiada con {len(df)} registros.")

✅ Tabla 'events' copiada con 13634 registros.
✅ Tabla 'users' copiada con 2131 registros.


In [37]:
(
    events
        .groupby('event_name',as_index=False)['user_id'].nunique()
)

,event_name,user_id
0,car_checkout,1056
1,payment_confirmation,676
2,product_page,1491
3,search_page,1716
4,sesion_start,2131
